# Ejercicios Avanzados: NumPy, Clases, FFT e I/O
*Observatorio Astronómico Nacional - Universidad Nacional de Colombia*  
*Estudiante:* Brayan Nicolas Buitrago Montenegro

### Ejercicio 1: Carga de datos + Preprocesamiento con NumPy

In [ ]:
import numpy as np

# El archivo se descargó mediante `curl -sO https://raw.githubusercontent.com/csmastersUH/data_analysis_with_python_2020/master/kumpula-weather-2017.csv`
# Nota: La estructura del archivo es Year, m, d, Time, Time zone, Precipitation amount (mm), Snow depth (cm), Air temperature (degC)


# PASO 1: Cargar el archivoo
# El archivo tiene una fila de encabezado (nombres de columnas),
# se usa names=True para que numpy lea esos nombres, también se podria ignorar el encabezado.
# delimiter="," porque es un CSV separado por comas.
datos = np.genfromtxt("kumpula-weather-2017.csv", delimiter=",", names=True)

# Vemos los nombres de las columnas que numpy detectó
print(datos.dtype.names)


('Year', 'm', 'd', 'Time', 'Time_zone', 'Precipitation_amount_mm', 'Snow_depth_cm', 'Air_temperature_degC')


In [ ]:
#Conrrect, lo leyo bien

# Se sacan las columnas que nos importan: mes, día y temperatura
meses = datos["m"]
dias_mes = datos["d"]
temperaturas = datos["Air_temperature_degC"]


# PASO 2: Función para convertir (mes, día) en "día del año"

def monthday_to_days(mes, dia):
    """
    Convierte mes y dia en el numero de dias transcurridos
    desde el 1 de enero (dia 0).
    Ej: 1 de enero -> 0, 2 de enero -> 1, 1 de febrero -> 31
    """
    # dias acumulados ANTES de empezar cada mes
    dias_por_mes = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    dias_acumulados = np.cumsum([0] + dias_por_mes[:-1]) # dias_por_mes[:-1] ignora el ultimo mes (diciembre)
    # dias_acumulados[mes-1] nos da los dias acumulados desde el 1 de enero hasta el mes anterior al actual.
    # antes de ese mes. Le sumamos (dia - 1) para no contar el dia actual dos veces.
    return dias_acumulados[int(mes) - 1] + (dia - 1)


# Aplicamos la funcion a cada fila 
days = np.array([monthday_to_days(m, d) for m, d in zip(meses, dias_mes)]) # zip() crea pares (mes, día)


# PASO 3: Filtrar NaN e inf
# np.isfinite() dice True si el numero es normal (no NaN, no inf)
mascara_validos = np.isfinite(temperaturas)

days_validos = days[mascara_validos]
temp_validos = temperaturas[mascara_validos]

print("Datos originales:", len(temperaturas))
print("Datos validos:", len(temp_validos))



# PASO 4: Media movil de 30 dias

def media_movil(valores, ventana=30):
    """
    Calcula el promedio de los ultimos 'ventana' dias 
    para cada punto. Usamos convolve, que es un truco de
    numpy para sumar "ventanas" que se deslizan.
    """
    pesos = np.ones(ventana) / ventana
    # mode="same" hace que el resultado tenga el mismo tamano
    # que la entrada
    return np.convolve(valores, pesos, mode="same")


media_30d = media_movil(temp_validos, 30)

print("Media movil de 30 dias calculada, forma:", media_30d.shape)

# PASO 5: Anomalias = temperatura real - media movil

anomalias = temp_validos - media_30d



# PASO 6: Detectar valores atipicos (outliers) con el metodo IQR. El IQR mide la dispersión de los datos

Q1 = np.percentile(temp_validos, 25) #primer cuartil)
Q3 = np.percentile(temp_validos, 75) #(tercer cuartil)
IQR = Q3 - Q1 #Rango Intercuartílico

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

mascara_outliers = (temp_validos < limite_inferior) | (temp_validos > limite_superior)


print("Cantidad de outliers encontrados:", np.sum(mascara_outliers))
print("Limite inferior:", limite_inferior, "Limite superior:", limite_superior)



# RESULTADO FINAL: temps_clean (N x 2) y days (N,)

# Usamos column_stack para pegar dos columnas (dias, temperatura)
# una al lado de la otra, formando una matriz de N filas x 2 columnas
temps_clean = np.column_stack((days_validos, temp_validos))
days = days_validos  # renombramos para que coincida

print("Forma de temps_clean:", temps_clean.shape)
print("Forma de days:", days.shape)


Datos originales: 365
Datos validos: 365
Media movil de 30 dias calculada, forma: (365,)
Cantidad de outliers encontrados: 2
Limite inferior: -16.35 Limite superior: 30.450000000000003
Forma de temps_clean: (365, 2)
Forma de days: (365,)


### Ejercicio 2: Clase avanzada con herencia y decoradores

In [ ]:
import numpy as np
import functools


# PASO 1: El decorador @vectorize

# Un decorador es una funcion que "envuelve" a otra funcion
# para agregarle un comportamiento extra, sin tener que
# reescribir esa funcion.
#
# Aqui, @vectorize va a asegurarse de que la funcion que
# decoramos siempre reciba un np.array (por si alguien le
# pasa una lista normal de python, la convertimos).
def vectorize(func):
    @functools.wraps(func)  # esto conserva el nombre original de la funcion
    def wrapper(self, columna, *args, **kwargs):
        # nos aseguramos de que 'columna' sea un array de numpy
        columna = np.asarray(columna, dtype=float)
        return func(self, columna, *args, **kwargs)
    return wrapper


# PASO 2: Clase base TimeSeriesAnalyzer

class TimeSeriesAnalyzer:
    """
    Clase 'padre' con funciones genericas para cualquier
    serie de tiempo (no solo clima).
    """

    def __init__(self, days, values):
        # self.days y self.values quedan guardados dentro
        # del objeto, para poder usarlos en otros metodos
        self.days = np.asarray(days, dtype=float)
        self.values = np.asarray(values, dtype=float)

    @vectorize
    def smooth(self, columna, window=7):
        """
        Media movil simple: promedia cada punto con sus
        vecinos, usando una 'ventana' de tamano 'window'.
        """
        pesos = np.ones(window) / window
        return np.convolve(columna, pesos, mode="same")



# PASO 3: Clase hija WeatherAnalyzer (hereda de TimeSeriesAnalyzer)

class WeatherAnalyzer(TimeSeriesAnalyzer):
    """
    Clase 'hija'. Como hereda de TimeSeriesAnalyzer, ya tiene
    gratis el metodo smooth() y el __init__, y aqui le
    agregamos funciones especificas del clima.
    """

    def seasonal_decompose(self):
        """
        Separa la serie de temperaturas en 3 partes:
        - tendencia: hacia donde va en general (ajuste polinomico grado 2)
        - estacional: el patron que se repite (usando FFT)
        - residuo: lo que sobra (ruido / lo que no explican las otras 2)
        """
        dias = self.days
        temps = self.values

        # --- Tendencia: ajustamos un polinomio de grado 2 ---
        # np.polyfit nos da los coeficientes de la "mejor curva"
        # que se ajusta a los datos
        coeficientes = np.polyfit(dias, temps, deg=2)
        # np.poly1d convierte esos coeficientes en una funcion
        # que podemos evaluar
        polinomio = np.poly1d(coeficientes)
        tendencia = polinomio(dias)

        # --- Quitamos la tendencia para quedarnos con lo que sobra ---
        sin_tendencia = temps - tendencia

        # --- Estacional: usamos FFT (transformada de Fourier) ---
        # FFT descompone la señal en "ondas" de distintas frecuencias.
        # Nos quedamos solo con las 4 frecuencias mas fuertes (las que
        # mas se repiten) para reconstruir el patron estacional.
        n = len(sin_tendencia)
        fft_valores = np.fft.fft(sin_tendencia)
        # magnitud de cada frecuencia (que tan "fuerte" es)
        magnitudes = np.abs(fft_valores)

        # ordenamos los indices de mayor a menor magnitud
        # (ignorando el indice 0, que es solo el promedio)
        indices_ordenados = np.argsort(magnitudes[1:])[::-1] + 1
        top4_indices = indices_ordenados[:4]

        # armamos una copia del fft donde solo dejamos esas 4
        # frecuencias principales (y sus 'espejos', porque la FFT
        # es simetrica), el resto lo ponemos en cero
        fft_filtrado = np.zeros(n, dtype=complex)
        for idx in top4_indices:
            fft_filtrado[idx] = fft_valores[idx]
            fft_filtrado[n - idx] = fft_valores[n - idx]  # frecuencia espejo

        # volvemos del "mundo de frecuencias" al "mundo normal"
        # con la transformada inversa. Usamos .real porque puede
        # quedar una parte imaginaria minuscula por errores de redondeo
        estacional = np.fft.ifft(fft_filtrado).real

        # --- Residuo: lo que no explican ni la tendencia ni lo estacional ---
        residuo = temps - tendencia - estacional

        return tendencia, estacional, residuo

    def forecast(self, days_ahead=30):
        """
        Pronostico simple (no es un ARIMA real, es una version
        'casera' parecida): usamos los ultimos 30 dias de datos,
        les ajustamos un polinomio de grado 3, y extendemos esa
        curva hacia el futuro, agregandole un poco de ruido
        aleatorio para que parezca mas realista.
        """
        # tomamos los ultimos 30 puntos de datos que tenemos
        ultimos_dias = self.days[-30:]
        ultimos_valores = self.values[-30:]

        # ajustamos un polinomio de grado 3 a esos ultimos 30 dias
        coeficientes = np.polyfit(ultimos_dias, ultimos_valores, deg=3)
        polinomio = np.poly1d(coeficientes)

        # generamos los dias futuros: desde el dia siguiente al
        # ultimo que tenemos, hasta 'days_ahead' dias mas
        ultimo_dia = self.days[-1]
        dias_futuros = np.arange(ultimo_dia + 1, ultimo_dia + 1 + days_ahead)

        # evaluamos el polinomio en esos dias futuros
        prediccion_base = polinomio(dias_futuros)

        # le sumamos un poco de "ruido" aleatorio, para que no
        # sea una linea perfecta (esto simula la incertidumbre)
        ruido = np.random.normal(loc=0, scale=1.0, size=days_ahead)
        prediccion_final = prediccion_base + ruido

        return dias_futuros, prediccion_final



# PASO 4: Uso de la clase (con los datos del Ejercicio 1)

# days y temps_clean vienen del Ejercicio 1
# temps_clean[:,1] es la SEGUNDA columna (la de temperaturas)
analyzer = WeatherAnalyzer(days, temps_clean[:, 1])

# probamos el suavizado (heredado de la clase base)
suavizado = analyzer.smooth(temps_clean[:, 1], window=7)
print("Primeros 10 valores suavizados:", suavizado[:10])

# probamos la descomposicion estacional
tendencia, estacional, residuo = analyzer.seasonal_decompose()
print("Tendencia (primeros 5):", tendencia[:5])
print("Estacional (primeros 5):", estacional[:5])
print("Residuo (primeros 5):", residuo[:5])

# probamos el pronostico
dias_futuros, prediccion = analyzer.forecast(days_ahead=30)
print("Dias futuros:", dias_futuros[:5], "...")
print("Prediccion:", prediccion[:5], "...")

Primeros 10 valores suavizados: [-3.22857143 -5.77142857 -8.31428571 -8.85714286 -9.01428571 -8.38571429
 -7.21428571 -5.61428571 -3.47142857 -0.77142857]
Tendencia (primeros 5): [-9.70591551 -9.48320988 -9.26160766 -9.04110886 -8.82171348]
Estacional (primeros 5): [2.15735696 2.19759599 2.23444137 2.26783184 2.29771047]
Residuo (primeros 5): [  8.14855854   3.38561388   0.52716629  -6.02672298 -11.27599699]
Dias futuros: [365. 366. 367. 368. 369.] ...
Prediccion: [-0.77951888  3.28927277  3.81456138  5.08307645  5.78218835] ...


### Ejercicio 3: I/O robusta + datos procesados

In [ ]:
import numpy as np
import os


# Recordatorio de datos que ya tenemos de los ejercicios 1 y 2

# temps_clean -> array (N, 2): columna 0 = dias, columna 1 = temperatura
# days        -> array (N,)
# anomalias   -> array (N,): temperatura - media movil de 30 dias
# suavizado   -> array (N,): temperatura suavizada (media movil de 7 dias)

# Por si acaso, nos aseguramos de tener nombres consistentes
anomalies = anomalias          # viene del Ejercicio 1
temp_smooth = suavizado        # viene del Ejercicio 2 (analyzer.smooth)



# PASO 1: Guardar temps_clean y anomalies en un archivo .npz

# np.savez guarda VARIOS arrays dentro de un solo archivo.
# Cada uno se guarda con un "nombre" (como si fuera un diccionario).
np.savez(
    "processed_weather.npz",
    temps=temps_clean,
    anomalies=anomalies
)

print("Archivo processed_weather.npz guardado correctamente.")



# PASO 2: Guardar un subconjunto (primeros 100 dias) en .csv

# Armamos una matriz con 3 columnas: dias, temperatura suavizada, anomalia
# Usamos [:100] para quedarnos solo con los primeros 100 valores de cada uno
subset = np.column_stack([
    days[:100],
    temp_smooth[:100],
    anomalies[:100]
])

# np.savetxt escribe un array en un archivo de texto plano (csv)
# - delimiter=',' -> separa los valores con comas
# - header='...'  -> escribe una primera linea con los nombres de columna
# - fmt='%.2f'    -> cada numero se escribe con 2 decimales
# - comments=''   -> para que el encabezado no empiece con '#' (por defecto numpy le pone '#')
np.savetxt(
    "subset.csv",
    subset,
    delimiter=",",
    header="days,temp_smooth,anomaly",
    fmt="%.2f",
    comments=""
)

print("Archivo subset.csv guardado correctamente.")



# PASO 3: Funcion load_and_validate

def load_and_validate(filename):
    """
    Carga un archivo .npz o .csv y revisa que los datos
    esten "sanos": sin NaN, sin infinitos, y con la forma
    (numero de columnas) esperada.

    Devuelve un diccionario con los datos y el resultado
    de la validacion.
    """
    resultado = {}

    # sacamos la extension del archivo (".npz" o ".csv")
    _, extension = os.path.splitext(filename)

    if extension == ".npz":
        # ---- Caso archivo .npz ----
        datos = np.load(filename)

        # datos.files nos da la lista de nombres guardados
        # (en nuestro caso: 'temps' y 'anomalies')
        for nombre in datos.files:
            arreglo = datos[nombre]

            # Verificamos que TODOS los valores sean finitos
            # (ni NaN ni inf). np.isfinite(arreglo).all() da
            # True solo si TODOS los valores son validos.
            es_finito = np.isfinite(arreglo).all()

            resultado[nombre] = {
                "data": arreglo,
                "shape": arreglo.shape,
                "all_finite": bool(es_finito)
            }

    elif extension == ".csv":
        # ---- Caso archivo .csv ----
        # skip_header=1 porque la primera fila es el encabezado (texto)
        arreglo = np.loadtxt(filename, delimiter=",", skiprows=1)

        es_finito = np.isfinite(arreglo).all()

        # Verificamos que tenga 3 columnas (days, temp_smooth, anomaly)
        forma_correcta = (arreglo.ndim == 2 and arreglo.shape[1] == 3)

        resultado["csv_data"] = {
            "data": arreglo,
            "shape": arreglo.shape,
            "all_finite": bool(es_finito),
            "shape_ok": forma_correcta
        }

    else:
        # Si no es .npz ni .csv, avisamos que no sabemos manejarlo
        raise ValueError(f"Formato no soportado: {extension}")

    return resultado



# PASO 4: Probamos la funcion con los dos archivos que creamos

resultado_npz = load_and_validate("processed_weather.npz")
print("\n--- Validacion de processed_weather.npz ---")
for nombre, info in resultado_npz.items():
    print(f"{nombre}: forma={info['shape']}, todos finitos={info['all_finite']}")

resultado_csv = load_and_validate("subset.csv")
print("\n--- Validacion de subset.csv ---")
for nombre, info in resultado_csv.items():
    print(f"{nombre}: forma={info['shape']}, todos finitos={info['all_finite']}, forma_ok={info['shape_ok']}")

Archivo processed_weather.npz guardado correctamente.
Archivo subset.csv guardado correctamente.

--- Validacion de processed_weather.npz ---
temps: forma=(365, 2), todos finitos=True
anomalies: forma=(365,), todos finitos=True

--- Validacion de subset.csv ---
csv_data: forma=(100, 3), todos finitos=True, forma_ok=True


## Pregunta 1: Outliers con IQR

**¿Qué fracción de los datos se elimina? ¿Es conservador o agresivo?**

El método IQR (`Q1 - 1.5*IQR` a `Q3 + 1.5*IQR`) suele eliminar entre 0.5% y 2% de los datos en distribuciones similares a la normal. Es un punto intermedio: ni muy agresivo ni muy permisivo.

- Si el porcentaje calculado es **menor a 1%**, es **conservador**.
- Si supera el **5%**, sería **agresivo** para datos climáticos (donde olas de frío/calor son reales, no errores).

**Alternativa propuesta: método sigma (3 desviaciones estándar)**

Con `sigma=3` se espera dejar afuera solo ~0.3% de los datos en una distribución normal: es más conservador que el IQR clásico, útil para no perder eventos climáticos extremos reales.

In [28]:
# Fraccion de datos eliminados con IQR
n_total = len(temp_validos)
n_outliers = np.sum(mascara_outliers)
fraccion = n_outliers / n_total

print(f"Outliers encontrados: {n_outliers} de {n_total}")
print(f"Fraccion eliminada: {fraccion:.2%}")

# Alternativa: metodo sigma=3
media = np.mean(temp_validos)
sigma = np.std(temp_validos)

limite_inf_sigma = media - 3 * sigma
limite_sup_sigma = media + 3 * sigma

mascara_outliers_sigma = (temp_validos < limite_inf_sigma) | (temp_validos > limite_sup_sigma)

n_outliers_sigma = np.sum(mascara_outliers_sigma)
print(f"\nOutliers con metodo sigma=3: {n_outliers_sigma}")
print(f"Fraccion eliminada (sigma=3): {n_outliers_sigma/n_total:.2%}")

Outliers encontrados: 2 de 365
Fraccion eliminada: 0.55%

Outliers con metodo sigma=3: 2
Fraccion eliminada (sigma=3): 0.55%


## Pregunta 2: Análisis de `@vectorize`

**Para `data.shape = (365, 2)`, ¿cuántas llamadas se hacen a `smooth(col)`?**

Con 2 columnas, el bucle `for` dentro del decorador llama a `smooth(col)` exactamente **2 veces** (una por cada columna: `data[:,0]` y `data[:,1]`).

**¿Por qué `results.T`?**

`np.array(resultados)` arma la salida con forma `(2, 365)` (2 filas de 365 elementos, una fila por cada columna procesada). Se usa **`.T`** (transponer) para volver a la forma original `(365, 2)`: filas = días, columnas = variables.

**¿Qué otro método de vectorización se propone?**

Usar `np.apply_along_axis`, que hace lo mismo "por dentro" (sigue siendo un bucle disimulado, no es más rápido). La vectorización realmente eficiente sería reescribir `smooth` para operar sobre toda la matriz de una sola vez con operaciones matriciales de NumPy, sin ningún `for` explícito.

In [30]:
import functools

def vectorize(func):
    @functools.wraps(func)
    def wrapper(self, data, *args, **kwargs):
        data = np.asarray(data, dtype=float)

        if data.ndim == 1:
            return func(self, data, *args, **kwargs)

        resultados = []
        for i in range(data.shape[1]):
            columna = data[:, i]
            resultado_columna = func(self, columna, *args, **kwargs)
            resultados.append(resultado_columna)

        # np.array(resultados) queda (columnas, filas) -> .T lo vuelve (filas, columnas)
        return np.array(resultados).T

    return wrapper

# Alternativa con apply_along_axis
# resultado = np.apply_along_axis(lambda col: analyzer.smooth(col), axis=0, arr=data)

## Pregunta 3: `seasonal_decompose()` — tendencia y frecuencias

**¿Por qué se resta la tendencia antes de la FFT?**

La FFT detecta patrones que se repiten (ondas). La tendencia (un cambio lento y sostenido, no repetitivo) contamina el análisis: la FFT la interpretaría como una "onda" de frecuencia muy baja, distorsionando la detección de los ciclos que sí se repiten. Por eso se resta primero (`sin_tendencia = temps - tendencia`).

**¿Qué representan las 4 frecuencias principales?**

- **Anual** (≈365 días): normalmente la más fuerte, representa el ciclo verano-invierno.
- **Mensual** (≈30 días): cambios de temporada dentro de una misma estación.
- **Semanal** (≈7 días): menos común en temperatura "natural", puede reflejar patrones de medición.
- **Diaria**: con una sola medición por día este dataset no puede captar el ciclo día/noche; probablemente no aparezca entre las 4 principales. Es una limitación real del dataset, vale la pena mencionarla.

## Pregunta 4: `analyzer.stats()` — original vs. suavizado (por columna)

**¿Cuál es el porcentaje de reducción de la desviación estándar por columna?**

`stats()` compara, para cada columna (por ejemplo temperatura y anomalía), 
la desviación estándar de los datos originales contra la de los datos 
suavizados con media móvil de 7 días. La media móvil "achata" picos y 
valles bruscos, por lo que se espera una reducción de la variabilidad, 
normalmente entre 10% y 40% según qué tan ruidosa sea cada columna.

**¿Por qué `polyfit` de grado 2 captura bien la tendencia?**

Un polinomio de grado 2 es una parábola: sube y baja (o al revés) una 
sola vez. La tendencia anual de temperatura suele tener justo esa forma 
(fría → cálida → fría). Un grado 2 alcanza para capturar esa curva 
general sin sobreajustarse a cada zigzag diario, que es justo lo que 
queremos dejar para el componente estacional y el residuo. Un grado 
mucho más alto empezaría a "perseguir" el ruido diario en vez de 
mostrar la tendencia real (sobreajuste).

In [33]:
class WeatherAnalyzer(TimeSeriesAnalyzer):

    def seasonal_decompose(self):
        dias = self.days
        temps = self.values

        coeficientes = np.polyfit(dias, temps, deg=2)
        polinomio = np.poly1d(coeficientes)
        tendencia = polinomio(dias)

        sin_tendencia = temps - tendencia

        n = len(sin_tendencia)
        fft_valores = np.fft.fft(sin_tendencia)
        magnitudes = np.abs(fft_valores)

        indices_ordenados = np.argsort(magnitudes[1:])[::-1] + 1
        top4_indices = indices_ordenados[:4]

        fft_filtrado = np.zeros(n, dtype=complex)
        for idx in top4_indices:
            fft_filtrado[idx] = fft_valores[idx]
            fft_filtrado[n - idx] = fft_valores[n - idx]

        estacional = np.fft.ifft(fft_filtrado).real
        residuo = temps - tendencia - estacional

        return tendencia, estacional, residuo

    def forecast(self, days_ahead=30):
        ultimos_dias = self.days[-30:]
        ultimos_valores = self.values[-30:]

        coeficientes = np.polyfit(ultimos_dias, ultimos_valores, deg=3)
        polinomio = np.poly1d(coeficientes)

        ultimo_dia = self.days[-1]
        dias_futuros = np.arange(ultimo_dia + 1, ultimo_dia + 1 + days_ahead)

        prediccion_base = polinomio(dias_futuros)
        ruido = np.random.normal(loc=0, scale=1.0, size=days_ahead)
        prediccion_final = prediccion_base + ruido

        return dias_futuros, prediccion_final

    def stats(self, data_original=None, nombres_columnas=None):
        """
        Compara la desviacion estandar ORIGINAL vs SUAVIZADA,
        columna por columna.
        """
        if data_original is None:
            data_original = self.values.reshape(-1, 1)  # (N,) -> (N,1)

        data_original = np.atleast_2d(data_original)
        if data_original.shape[0] == 1:
            data_original = data_original.T

        n_columnas = data_original.shape[1]

        if nombres_columnas is None:
            nombres_columnas = [f"columna_{i}" for i in range(n_columnas)]

        resultados = {}

        for i in range(n_columnas):
            columna_original = data_original[:, i]
            columna_suavizada = self.smooth(columna_original, window=7)

            std_original = np.std(columna_original)
            std_suavizado = np.std(columna_suavizada)
            reduccion_pct = (1 - std_suavizado / std_original) * 100

            nombre = nombres_columnas[i]
            resultados[nombre] = {
                "std_original": std_original,
                "std_smooth": std_suavizado,
                "reduccion_pct": reduccion_pct
            }

            print(f"--- {nombre} ---")
            print(f"  Desviacion estandar original:  {std_original:.3f}")
            print(f"  Desviacion estandar suavizada: {std_suavizado:.3f}")
            print(f"  Reduccion de la variabilidad:  {reduccion_pct:.2f}%\n")

        return resultados

In [34]:
# Volvemos a crear el objeto con la clase actualizada
analyzer = WeatherAnalyzer(days, temps_clean[:, 1])

# Comparamos temperatura y anomalia, columna por columna
datos_para_stats = np.column_stack([temps_clean[:, 1], anomalies])

resultado_stats = analyzer.stats(
    data_original=datos_para_stats,
    nombres_columnas=["temperatura", "anomalia"]
)

--- temperatura ---
  Desviacion estandar original:  7.174
  Desviacion estandar suavizada: 6.823
  Reduccion de la variabilidad:  4.89%

--- anomalia ---
  Desviacion estandar original:  2.825
  Desviacion estandar suavizada: 1.774
  Reduccion de la variabilidad:  37.22%

